In [135]:
import pandas as pd
import numpy as np
from unidecode import unidecode
from pathlib import Path
import re
import hashlib

pd.set_option('future.no_silent_downcasting', True) # Evita avisos de downcasting

# Buscar a pasta onde está o parquet
pasta = Path("../data/01_bronze/Notificacoes")
arquivo_parquet = next(pasta.glob("*.parquet"))
df = pd.read_parquet(arquivo_parquet)

In [136]:
# Normalizar as linhas
def normalize_rows(series):
    series = series.astype(str).str.strip()  # limpa espaços
    series = series.replace(["nan", "None", "NaT", "NULL"], np.nan)  # textos inválidos
    series = series.apply(lambda x: unidecode(x) if pd.notna(x) else x)
    series = series.str.upper()
    return series

for col in df.select_dtypes(include=['object']).columns:
    df[col] = normalize_rows(df[col])

In [137]:
# Converter datas no formato YYYY-MM-DD
def normalize_date_column(series):
    # Converte strings ou números para datetime
    series = pd.to_datetime(series.astype(str).str.strip(), errors="coerce", format="%Y%m%d")
    # Mantém apenas a parte da data (sem hora)
    series = series.dt.floor("d")  # arredonda para o dia
    return series

colunas_data = [
    "DATA_INCLUSAO_SISTEMA", "DATA_ULTIMA_ATUALIZACAO", "DATA_NOTIFICACAO",
    "DATA_NASCIMENTO", "DATA_INICIO_HORA", "DATA_FINAL_HORA"
]

for col in colunas_data:
    if col in df.columns:
        df[col] = normalize_date_column(df[col])

In [138]:
def mapear_coluna(df, coluna, mapa, nome_valor=None, tipo_int64=True):
    if coluna not in df.columns:
        return df

    nome_valor = nome_valor or f"{coluna}_VALOR"
    df[nome_valor] = df[coluna]

    # Substitui pelos códigos
    df[coluna] = df[coluna].replace(mapa)
    
    if tipo_int64:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0).astype("Int64")

    return df

In [139]:
# Mapeamento UF numérico
uf_map = {
    "AC": 1, "AL": 2, "AP": 3, "AM": 4, "BA": 5, "CE": 6, "DF": 7, "ES": 8, "GO": 9, "MA": 10, 
    "MT": 11, "MS": 12, "MG": 13, "PA": 14, "PB": 15, "PR": 16, "PE": 17, "PI": 18, "RJ": 19,
    "RN": 20, "RS": 21, "RO": 22, "RR": 23, "SC": 24, "SP": 25, "SE": 26, "TO": 27
}

uf_nome_map = {
    "AC": "ACRE", "AL": "ALAGOAS", "AP": "AMAPA", "AM": "AMAZONAS", "BA": "BAHIA", "CE": "CEARA",
    "DF": "DISTRITO FEDERAL", "ES": "ESPIRITO SANTO", "GO": "GOIAS", "MA": "MARANHAO", "MT": "MATO GROSSO",
    "MS": "MATO GROSSO DO SUL", "MG": "MINAS GERAIS", "PA": "PARA", "PB": "PARAIBA", "PR": "PARANA",
    "PE": "PERNAMBUCO", "PI": "PIAUI", "RJ": "RIO DE JANEIRO", "RN": "RIO GRANDE DO NORTE",
    "RS": "RIO GRANDE DO SUL", "RO": "RONDONIA", "RR": "RORAIMA", "SC": "SANTA CATARINA",
    "SP": "SAO PAULO", "SE": "SERGIPE", "TO": "TOCANTINS"
}

if "UF" in df.columns:
    df["UF_VALOR"] = df["UF"]
    df = mapear_coluna(df, "UF", uf_map)
    df["UF_DESCRICAO"] = df["UF_VALOR"].map(uf_nome_map).fillna("DESCONHECIDO")

In [140]:
# Mapeamento GRUPO_IDADE numérico
grupo_idade_map = {
    "FETO": 1, "NEONATO": 2, "INFANTIL": 3, "CRIANCA": 4, "ADOLESCENTE": 5,
    "ADULTO": 6, "IDOSO": 7, "OUTRO": 8, "NAO INFORMADO": 0
}

grupo_idade_descricao = {
    "FETO": "PRE-NASCIMENTO", "NEONATO": "0 A 27 DIAS", "INFANTIL": "28 DIAS A 1 ANO",
    "CRIANCA": "2 A 11 ANOS", "ADOLESCENTE": "12 A 18 ANOS", "ADULTO": "19 A 59 ANOS",
    "IDOSO": "60 ANOS OU MAIS", "OUTRO": "DESCONHECIDO", "NAO INFORMADO": "DESCONHECIDO"
}

if "GRUPO_IDADE" in df.columns:
    df["GRUPO_IDADE_VALOR"] = df["GRUPO_IDADE"]
    df = mapear_coluna(df, "GRUPO_IDADE", grupo_idade_map)
    df["GRUPO_IDADE_DESCRICAO"] = df["GRUPO_IDADE_VALOR"].map(grupo_idade_descricao).fillna("DESCONHECIDO")

In [141]:
# Mapear colunas SIM/NAO
def normalize_sim_nao(df, coluna):
    if coluna not in df.columns:
        return df

    df[coluna] = df[coluna].astype(str).str.strip().str.upper()
    df[coluna + "_VALOR"] = df[coluna].where(df[coluna].isin(["SIM", "NAO", "NAO INFORMADO"]), "NAO INFORMADO")

    mapa = {"SIM": 1, "NAO": 2, "NAO INFORMADO": 0}
    df[coluna] = pd.to_numeric(df[coluna + "_VALOR"].replace(mapa), errors="coerce").fillna(0).astype("Int64")
    return df

colunas_sim_nao = ["NOTIFICACAO_PARENT_CHILD", "GESTANTE", "LACTANTE", "GRAVE"]
for col in colunas_sim_nao:
    df = normalize_sim_nao(df, col)

In [142]:
def extrair_numero(df, coluna, nome_valor=None, nome_unidade=None):
    if coluna not in df.columns:
        return df

    nome_valor = nome_valor or f"{coluna}_VALOR"
    nome_unidade = nome_unidade or f"{coluna}_UNIDADE"

    def extrair(valor):
        if pd.isna(valor):
            return (np.nan, np.nan)
        texto = str(valor).strip().upper()
        match = re.match(r"(\d+)\s*([A-ZÇÃÉÍ]+)?", texto)
        if not match:
            return (np.nan, np.nan)
        numero = int(match.group(1))
        unidade = match.group(2) if match.group(2) else ""
        unidade = ("ANO(S)" if "ANO" in unidade else "MES(ES)" if "MES" in unidade else
                  "DIA(S)" if "DIA" in unidade else "NAN")
        return (numero, unidade)

    df[[nome_valor, nome_unidade]] = df[coluna].apply(lambda x: pd.Series(extrair(x)))
    df[nome_valor] = df[nome_valor].fillna(0).astype("Int64")
    return df

df = extrair_numero(df, "IDADE_MOMENTO_REACAO")
df = extrair_numero(df, "DURACAO")
df = extrair_numero(df, "IDADE_GESTACIONAL_MOMENTO_REACAO")

In [143]:
def inferir_tipos(df):
    tipos = {}
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            tipos[col] = "Int64" if pd.api.types.is_integer_dtype(df[col]) else "float64"
        elif pd.api.types.is_object_dtype(df[col]):
            sample = df[col].dropna().head(20).astype(str)
            if all(pd.to_datetime(sample, format="%Y%m%d", errors="coerce").notna()):
                tipos[col] = "datetime64[ns]"
            else:
                tipos[col] = "string"
        else:
            tipos[col] = str(df[col].dtype)
    return tipos

tipos_detectados = inferir_tipos(df)
for col, tipo in tipos_detectados.items():
    try:
        if "datetime" in tipo:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
        else:
            df[col] = df[col].astype(tipo)
    except Exception as e:
        pass

for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].replace(["", " ", "NAN", "NONE", "NULL"], np.nan)
    df[col] = df[col].fillna("DESCONHECIDO")

In [144]:
# TIPO_ENTRADA_VIGIMED
tipo_entrada_vigimed_map = {
    "SERVICOS DE SAUDE": 1, "EMPRESAS FARMACEUTICAS": 2, "PACIENTES E PROFISSIONAIS DE SAUDE": 3,
    "SERVICOS DE VACINACAO": 4, "EREPORTING - INDUSTRIA, ENTRADA MANUAL DE DADOS": 5,
    "EREPORTING - INDUSTRIA, CARGA E2B": 6, "VIGIMOBILE": 7, "VIGIFLOW EFORMS": 8
}

# RECEBIDO_DE
recebido_de_map = {
    "CARGA E2B": 1, "ENTRADA MANUAL DE DADOS": 2, "AUTORIDADE REGULADORA": 3,
    "CENTRO REGIONAL DE FARMACOVIGILANCIA": 4, "EMPRESA FARMACEUTICA": 5,
    "OUTRO (P.EX. DISTRIBUIDORA, FINANCIADOR DE ESTUDO, ORGANIZACAO REPRESENTATIVA PARA PESQUISA CLINICA, OU ORGANIZACAO NAO-COMERCIAL)": 6,
    "PACIENTE/CONSUMIDOR": 7, "PROFISSIONAL DE SAUDE": 8
}

# TIPO_NOTIFICACAO
tipo_notificacao_map = {
    "NOTIFICACAO ESPONTANEA": 1, "NOTIFICACAO DE ESTUDO": 2, "OUTRO": 3,
    "NAO DISPONIVEL PELO NOTIFICADOR (DESCONHECIDO)": 4
}

# SEXO
sexo_map = {"FEMININO": 1, "MASCULINO": 2, "DESCONHECIDO": 0}

# GRAVIDADE
gravidade_map = { 
    "OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO": 1, "HOSPITALIZACAO": 2, "AMEACA R VIDA": 3,
    "RESULTOU EM OBITO": 4, "INCAPACIDADE PERSISTENTE OU SIGNIFICATIVA": 5,
    "ANOMALIA CONGENITA OU MALFORMACAO AO NASCER": 6, "NAO INFORMADO": 0
} 

# DESFECHO
desfecho_map = { 
    "RECUPERADO": 1, "DESCONHECIDO": 2, "EM RECUPERACAO": 4, "NAO RECUPERADO": 5, "FATAL": 6
} 

# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1, "CONCOMITANTE": 2, "INTERACAO": 3, "MEDICAMENTO NAO ADMINISTRADO": 4
}

# ACAO_ADOTADA
acao_adotada_map = {
    "AUMENTO DA DOSE": 1, "DESCONHECIDO": 2, "NAO APLICAVEL": 3,
    "REDUCAO DA DOSE": 4, "RETIRADA DO MEDICAMENTO": 5, "SEM ALTERACAO DA DOSE": 6
}

# NOTIFICADOR
notificador_map = {
    "ADVOGADO": 1, "CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE": 2,
    "FARMACEUTICO": 3, "MEDICO": 4, "OUTRO PROFISSIONAL DE SAUDE": 5
}

df = mapear_coluna(df, "TIPO_ENTRADA_VIGIMED", tipo_entrada_vigimed_map)
df = mapear_coluna(df, "RECEBIDO_DE", recebido_de_map)
df = mapear_coluna(df, "TIPO_NOTIFICACAO", tipo_notificacao_map)
df = mapear_coluna(df, "SEXO", sexo_map)
df = mapear_coluna(df, "GRAVIDADE", gravidade_map)
df = mapear_coluna(df, "DESFECHO", desfecho_map)
df = mapear_coluna(df, "RELACAO_MEDICAMENTO_EVENTO", relacao_medicamento_evento_map)
df = mapear_coluna(df, "ACAO_ADOTADA", acao_adotada_map)
df = mapear_coluna(df, "NOTIFICADOR", notificador_map)

In [145]:
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].replace(["", " ", "NAN", "NONE", "NULL"], np.nan)
    df[col] = df[col].fillna("DESCONHECIDO")

In [146]:
# Data e flags
data_atual = pd.Timestamp.now().strftime("%Y-%m-%d")

df["VALIDADE_REGISTRO_INICIAL"] = data_atual
df["VALIDADE_REGISTRO_FINAL"] = "9999-12-31"
df["ATUALIZADO_EM"] = data_atual
df["FLAG_ATUAL"] = True

In [147]:
# Hash único
def gerar_hash_linha(row):
    linha_str = "|".join("" if pd.isna(x) else str(x) for x in row.values)
    return hashlib.sha256(linha_str.encode("utf-8")).hexdigest()

df["HASH"] = df.apply(gerar_hash_linha, axis=1)

In [148]:
# Cria uma chave primária sequencial
df["PK_NOTIFICACAO"] = range(1, len(df) + 1)
df["PK_NOTIFICACAO"] = df["PK_NOTIFICACAO"].astype("int64")

In [149]:
# Organização de colunas
grupos = [
    ["PK_NOTIFICACAO"],
    ["UF", "UF_VALOR", "UF_DESCRICAO",
    "TIPO_ENTRADA_VIGIMED", "TIPO_ENTRADA_VIGIMED_VALOR",
    "RECEBIDO_DE", "RECEBIDO_DE_VALOR",
    "IDENTIFICACAO_NOTIFICACAO", 
    "DATA_INCLUSAO_SISTEMA", 
    "DATA_ULTIMA_ATUALIZACAO",
    "DATA_NOTIFICACAO", 
    "TIPO_NOTIFICACAO", "TIPO_NOTIFICACAO_VALOR",
    "NOTIFICACAO_PARENT_CHILD", "NOTIFICACAO_PARENT_CHILD_VALOR",
    "DATA_NASCIMENTO",
    "IDADE_MOMENTO_REACAO_VALOR", "IDADE_MOMENTO_REACAO_UNIDADE",
    "GRUPO_IDADE", "GRUPO_IDADE_VALOR", "GRUPO_IDADE_DESCRICAO",
    "IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR", "IDADE_GESTACIONAL_MOMENTO_REACAO_UNIDADE",
    "SEXO", "SEXO_VALOR",
    "GESTANTE", "GESTANTE_VALOR",
    "LACTANTE", "LACTANTE_VALOR",
    "PESO_KG", "ALTURA_CM",
    "REACAO_EVENTO_ADVERSO_MEDDRA",
    "GRAVE", "GRAVE_VALOR",
    "GRAVIDADE", "GRAVIDADE_VALOR",
    "DESFECHO", "DESFECHO_VALOR",
    "DATA_INICIO_HORA", "DATA_FINAL_HORA",
    "DURACAO_VALOR", "DURACAO_UNIDADE",
    "RELACAO_MEDICAMENTO_EVENTO", "RELACAO_MEDICAMENTO_EVENTO_VALOR",
    "NOME_MEDICAMENTO_WHODRUG",
    "ACAO_ADOTADA", "ACAO_ADOTADA_VALOR",
    "NOTIFICADOR", "NOTIFICADOR_VALOR",],
    ["HASH", "VALIDADE_REGISTRO_INICIAL", "VALIDADE_REGISTRO_FINAL", 
    "ATUALIZADO", "ATUALIZADO_EM", "FLAG_ATUAL"],
]

ordem_colunas = [col for grupo in grupos for col in grupo if col in df.columns]
colunas_restantes = [col for col in df.columns if col not in ordem_colunas]
df = df[ordem_colunas + colunas_restantes]

In [150]:
colunas_remover = ["IDADE_MOMENTO_REACAO", "IDADE_GESTACIONAL_MOMENTO_REACAO", "DURACAO", "ATUALIZADO"]
df = df.drop(columns=[col for col in colunas_remover if col in df.columns])

In [151]:
# Caminho do arquivo Gold
caminho_gold = "../data/03_gold/dim_notificacoes/dim_notificacoes.parquet"

# Data atual (datetime) para versionamento
data_atual = pd.Timestamp.now().floor("d")

try:
    df_antigo = pd.read_parquet(caminho_gold)

    # Garante que as colunas essenciais existam em ambos
    colunas_comuns = [c for c in df.columns if c in df_antigo.columns]
    df = df[colunas_comuns]
    df_antigo = df_antigo[colunas_comuns]

    # Mantém tipos consistentes
    if "PK_NOTIFICACAO" in df_antigo.columns:
        df_antigo["PK_NOTIFICACAO"] = pd.to_numeric(df_antigo["PK_NOTIFICACAO"], errors="coerce").fillna(0).astype("int64")

    # Detecta novos registros (novos HASH)
    novos_hashes = set(df["HASH"]) - set(df_antigo["HASH"])
    alterados = df[df["HASH"].isin(novos_hashes)]

    if len(alterados) > 0:
        df_antigo.loc[df_antigo["HASH"].isin(novos_hashes), "FLAG_ATUAL"] = False
        df_antigo.loc[df_antigo["HASH"].isin(novos_hashes), "VALIDADE_REGISTRO_FINAL"] = data_atual
        df = pd.concat([df_antigo, alterados], ignore_index=True)
    else:
        df = df_antigo.copy()

except FileNotFoundError:
    pass    

# Salvar gold
df.to_parquet(caminho_gold, index=False)